# Grupo 11 · E2 · EJECUCIÓN RÁPIDA (exploratoria)
**Abrir en Google Colab con GPU:** Runtime / Change runtime type / T4 GPU.

> **Atención:** Esta variante reduce los subconjuntos de entrenamiento y evaluación. Los resultados NO son la evaluación final de 10 000 imágenes ni sustituyen un experimento completo. Guardar cifras como «exploratorias».

El notebook E2 original permanece intacto. Este notebook permite detectar errores, generar primeras gráficas y comparar H1 con menor costo.


In [ ]:
# Celda 1: entorno
import sys, subprocess
# Colab suele incluir torch/torchvision, instalar solo dependencias livianas
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pandas', 'matplotlib'])
import os, json, time, random, copy, math, platform, pathlib, io
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from IPython.display import display
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), 'Se requiere GPU en Colab: Entorno de ejecución > Cambiar tipo > T4 GPU. No correr ViT-B/16 en CPU durante horas.'
DEVICE=torch.device('cuda')
ROOT=pathlib.Path('grupo11_outputs'); ROOT.mkdir(exist_ok=True)
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no disponible')


## 1. Dataset y adaptación del modelo
La evaluación de CIFAR-10 se realiza sobre el split oficial **test**, no sobre training. Se usa validación independiente extraída de train para elegir el checkpoint. Para primera ejecución rápida puede reducirse `N_TRAIN`, pero no se debe etiquetar esa corrida como el entrenamiento final. Se guardan los detalles de cada corrida.

In [ ]:
WEIGHTS=ViT_B_16_Weights.IMAGENET1K_V1
# La normalización coincide con pesos ImageNet V1; se redimensiona CIFAR-10 a 224×224
norm=transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
train_tf=transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
eval_tf=transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), norm])
raw_train=datasets.CIFAR10(root='./data',train=True,download=True,transform=train_tf)
raw_val=datasets.CIFAR10(root='./data',train=True,download=False,transform=eval_tf)
raw_test=datasets.CIFAR10(root='./data',train=False,download=True,transform=eval_tf)
rng=np.random.default_rng(SEED); perm=rng.permutation(len(raw_train))
val_ids=perm[:5000]; train_ids=perm[5000:]
N_TRAIN=3000  # EXPLORATORIO: para pruebas; usar 15000 o 45000 para experimentos más extensos
train_ids=train_ids[:N_TRAIN]
BATCH=16  # conservador para T4; subir a 32 solo si la VRAM alcanza
WORKERS=2 if DEVICE.type=='cuda' else 0
train_loader=DataLoader(Subset(raw_train, train_ids.tolist()),batch_size=BATCH,shuffle=True,num_workers=WORKERS,pin_memory=DEVICE.type=='cuda')
N_VAL_EVAL=750  # subconjunto fijo EXPLORATORIO de la partición de validación
N_TEST_EVAL=1000 # subconjunto fijo EXPLORATORIO; no equivale al test final
val_loader=DataLoader(Subset(raw_val,val_ids[:N_VAL_EVAL].tolist()),batch_size=BATCH,shuffle=False,num_workers=WORKERS)
test_loader=DataLoader(Subset(raw_test, list(range(N_TEST_EVAL))),batch_size=BATCH,shuffle=False,num_workers=WORKERS)
print('entrenamiento',len(train_ids),'validación',len(val_ids),'test',len(raw_test))
model=vit_b_16(weights=WEIGHTS)
model.heads.head=nn.Linear(model.heads.head.in_features, 10)
model.to(DEVICE)
print('parámetros:',sum(p.numel() for p in model.parameters()))


In [ ]:
# Fine-tuning de últimos 2 bloques y head. Evita entrenar 86M parámetros desde cero.
# Si se dispone de checkpoint YA entrenado, copiar a grupo11_outputs/vit_cifar10_best.pt y establecer TRAIN=False
TRAIN=True  # Si ya se guardó el checkpoint y quieres evitar reentrenar, usa False
EPOCHS=1  # corrida exploratoria; aumentar solo después de comprobar runtime
for p in model.parameters(): p.requires_grad=False
for block in list(model.encoder.layers.children())[-2:]:
    for p in block.parameters(): p.requires_grad=True
for p in model.heads.parameters(): p.requires_grad=True

def accuracy(model, loader, device=DEVICE):
    model.eval(); correct=total=0
    with torch.inference_mode():
        for xb,yb in loader:
            pred=model(xb.to(device)).argmax(1).cpu();correct+=(pred==yb).sum().item();total+=yb.numel()
    return 100*correct/total

from tqdm.auto import tqdm
# Montar Google Drive para que el checkpoint sobreviva a desconexiones
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR=pathlib.Path('/content/drive/MyDrive/Grupo11_ViT_checkpoints');CKPT_DIR.mkdir(parents=True,exist_ok=True)
ckpt=CKPT_DIR/'vit_cifar10_fast_best.pt'
print('Checkpoint persistente:',ckpt)
if TRAIN:
    opt=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-5,weight_decay=0.01)
    best=-1; history=[]
    for epoch in range(EPOCHS):
        model.train(); running=0
        for xb,yb in tqdm(train_loader, desc=f'Entrenamiento época {epoch+1}/{EPOCHS}'):
            opt.zero_grad(set_to_none=True)
            logits=model(xb.to(DEVICE));loss=nn.functional.cross_entropy(logits,yb.to(DEVICE))
            loss.backward();opt.step();running+=loss.item()
        val=accuracy(model,val_loader)
        history.append({'epoch':epoch+1,'val_top1':val,'sum_train_batch_losses':running})
        print('epoch',epoch+1,'validation top1',round(val,2))
        if val>best:
            best=val;torch.save(model.state_dict(),ckpt)
    pd.DataFrame(history).to_csv(ROOT/'train_history.csv',index=False)
assert ckpt.exists(), 'Falta checkpoint entrenado de CIFAR-10'
model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True));model.eval()
print('Mejor checkpoint:',ckpt)


## 2. Fase A: teardown
Calculamos parámetros de cada encoder block y FLOPs aproximados **analíticos** de Linear/Conv y multiplicaciones de matrices de atención. No se cuentan normalización, GELU, softmax, interpolación, ni overhead: por tanto **no son FLOPs exactos ni equivalentes a un profiler exhaustivo**. En TorchVision ViT-B/16: 197 tokens = 196 patches + CLS.

In [ ]:
# Parámetros por bloque / módulo
rows=[]
for i,block in enumerate(model.encoder.layers):
    mha=block.self_attention
    rows.append({'bloque':i+1,'params_totales':sum(p.numel() for p in block.parameters()),
                 'params_MHSA':sum(p.numel() for p in mha.parameters()),
                 'params_MLP':sum(p.numel() for p in block.mlp.parameters())})
params=pd.DataFrame(rows)
# Conteo aproximado de MACs y FLOPs por bloque con B=1, tokens=197, D=768, MLP=3072
N=197;D=768;H=3072
macs_qkv=3*N*D*D
macs_out=N*D*D
macs_attention=2*N*N*D
macs_mlp=2*N*D*H
block_flops=2*(macs_qkv+macs_out+macs_attention+macs_mlp)
params['FLOPs_aprox']=block_flops
params.to_csv(ROOT/'teardown_por_bloque.csv',index=False)
display(params)
print('Parámetros modelo:',sum(p.numel() for p in model.parameters()))
print('FLOPs aproximados de 12 encoder blocks:',12*block_flops/1e9,'GFLOPs (sin patch embedding/head)')


In [ ]:
# Activaciones intermedias a través de hooks y mapas aproximados de similitud CLS-patches
acts={}; handles=[]
for ix in [0,5,11]:
    block=list(model.encoder.layers)[ix]
    def mk(k):
        def cb(module, inputs, output): acts[k]=output.detach().cpu()
        return cb
    handles.append(block.register_forward_hook(mk(ix+1)))
xb,yb=next(iter(test_loader))
with torch.inference_mode(): _=model(xb[:1].to(DEVICE))
for h in handles: h.remove()
fig,axs=plt.subplots(1,3,figsize=(12,3.3))
for ax,(layer,activation) in zip(axs,acts.items()):
    # Similitud coseno de representación CLS frente a patches (no es mapa de atención exacto ni campo receptivo causal)
    t=activation[0]; scores=nn.functional.cosine_similarity(t[0:1],t[1:],dim=-1).reshape(14,14)
    ax.imshow(scores,interpolation='nearest');ax.set_title(f'Capa {layer}: similitud CLS');ax.axis('off')
plt.tight_layout();plt.savefig(ROOT/'features_intermedias.png',dpi=150);plt.show()
print('Estos mapas NO sustituyen una medición de campo receptivo efectivo.')


In [ ]:
# Campo de influencia efectivo: gradiente de logit de clase predicha sobre la imagen (proxy local)
model.eval()
x=xb[:1].to(DEVICE).clone().detach().requires_grad_(True)
with torch.enable_grad():
    out=model(x);target=out[0,out.argmax(1).item()]; grad=torch.autograd.grad(target,x)[0]
saliency=grad.abs().amax(dim=1)[0].detach().cpu()
plt.figure(figsize=(5,4));plt.imshow(saliency,cmap='magma');plt.title('Gradiente espacial |d logit / d pixel|');plt.colorbar();plt.axis('off')
plt.tight_layout();plt.savefig(ROOT/'campo_influencia_gradiente.png',dpi=150);plt.show()
print('Mapa de sensibilidad local al input, no campo receptivo teórico.')


In [ ]:
# Medición rigurosa de latencia: sincronizar CUDA y excluir warmup; mismo batch para todas las variantes
import statistics
@torch.inference_mode()
def profile(model, loader, device, batches=8, warmup=3):
    model.eval(); x=next(iter(loader))[0].to(device)
    for _ in range(warmup): _=model(x)
    if device.type=='cuda': torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats(device)
    times=[]
    for _ in range(batches):
        if device.type=='cuda':torch.cuda.synchronize()
        start=time.perf_counter();_=model(x)
        if device.type=='cuda':torch.cuda.synchronize()
        times.append((time.perf_counter()-start)*1000/x.shape[0])
    mem=torch.cuda.max_memory_allocated(device)/2**20 if device.type=='cuda' else None
    return {'latencia_ms_img_mediana':statistics.median(times),'latencia_ms_img_p90':float(np.percentile(times,90)),
            'VRAM_pico_MiB':mem,'batch':len(x),'device':str(device)}
def size_mb(model):
    b=io.BytesIO();torch.save(model.state_dict(),b);return len(b.getvalue())/1e6
baseline={'experimento':'FP32_baseline','tipo':'FP32 real','accuracy_top1_pct':accuracy(model,test_loader),
          'tamano_checkpoint_MB':size_mb(model),**profile(model,test_loader,DEVICE)}
print(baseline)


## 3. Fase B: primera intervención + ablación controlada
**H1 predicha antes de ejecutar:** la cuantización simulada de pesos de MHSA a INT8 perjudicará más accuracy que la cuantización simulada de pesos MLP a INT8. Ambas utilizan el mismo algoritmo simétrico per-tensor. **Se cuantizan y de-cuantizan los tensores a FP32**, por lo que el checkpoint conserva tamaño FP32. Se evalúa únicamente el efecto numérico, **no** los beneficios de despliegue. INT4 simulado es exploratorio, no un artefacto INT4 empaquetado.

**Importante:** INT8 dinámico de `nn.Linear` no cuantiza automáticamente el parámetro empacado `in_proj_weight` de `nn.MultiheadAttention` de TorchVision. Se evita denominarlo MHSA INT8 real.

In [ ]:
# PTQ simulado selectivo para MHSA o MLP: cuantiza y restaura pesos, no cambia arquitectura
@torch.no_grad()
def fake_quantize_tensor_(param, bits=8):
    max_abs=param.data.abs().max()
    if max_abs == 0: return
    qmax=(2**(bits-1))-1
    scale=max_abs/qmax
    quant=torch.clamp(torch.round(param.data/scale),-qmax,qmax)
    param.data.copy_(quant*scale)

def simulated_quant_model(base, target='mhsa', bits=8):
    clone=copy.deepcopy(base).to('cpu')
    for layer in clone.encoder.layers:
        module=layer.self_attention if target=='mhsa' else layer.mlp if target=='mlp' else None
        if module is None: raise ValueError(target)
        for name,p in module.named_parameters():
            if p.ndim>=2: fake_quantize_tensor_(p,bits)
    return clone.to(DEVICE).eval()

results=[baseline]
# Misma semilla, mismo test, mismo tamaño de batch y hardware
for bits,target in [(8,'mhsa'),(8,'mlp')]:
    variant=simulated_quant_model(model,target=target,bits=bits)
    row={'experimento':f'FakeINT{bits}_{target.upper()}',
         'tipo':f'simulación PTQ {bits} bits sobre pesos; almacenamiento FP32',
         'accuracy_top1_pct':accuracy(variant,test_loader),
         'tamano_checkpoint_MB':size_mb(variant),**profile(variant,test_loader,DEVICE)}
    results.append(row);print(row)
    del variant
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
df=pd.DataFrame(results)
df['caida_pp_vs_FP32']=baseline['accuracy_top1_pct']-df['accuracy_top1_pct']
df.to_csv(ROOT/'resultados_e2.csv',index=False)
display(df)


In [ ]:
# Gráficos de sensibilidad (NO presentar como curva de compresión real)
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(df.experimento,df.accuracy_top1_pct)
ax.set_ylabel('Accuracy Top-1 (%)'); ax.set_title('Ablación: efecto numérico fake PTQ INT8')
ax.tick_params(axis='x',rotation=20);plt.tight_layout();plt.savefig(ROOT/'ablacion_accuracy.png',dpi=160);plt.show()
print('Diferencia MHSA - MLP (puntos porcentuales):',float(df.loc[df.experimento=='FakeINT8_MHSA','accuracy_top1_pct'].iloc[0]-df.loc[df.experimento=='FakeINT8_MLP','accuracy_top1_pct'].iloc[0]))


## 4. FP16 real (GPU) y tamaño
Si existe CUDA se convierte a `half()` y se evalúa con entradas half; comparar con FP32 bajo la misma GPU. En CPU se omite explícitamente. Esto **sí** cambia precisión y almacenamiento, pero una mejora de latencia no está garantizada.

In [ ]:
if DEVICE.type=='cuda':
    # Evitar tener dos ViT simultáneos ocupando VRAM: ejecutar FP16 al final
    model.cpu();torch.cuda.empty_cache()
    half_model=copy.deepcopy(model).to(DEVICE).half().eval()
    @torch.inference_mode()
    def acc_half(m,loader):
        good=total=0
        for xx,yy in loader:
            preds=m(xx.to(DEVICE).half()).argmax(1).cpu()
            good+=(preds==yy).sum().item();total+=len(yy)
        return 100*good/total
    # wrapper ejecuta las mismas entradas en FP16
    class HalfInput(nn.Module):
        def __init__(self,net):super().__init__();self.net=net
        def forward(self,x):return self.net(x.half())
    half_wrapper=HalfInput(half_model).to(DEVICE).eval()
    f16={'experimento':'FP16_real','tipo':'pesos y activaciones FP16',
         'accuracy_top1_pct':acc_half(half_model,test_loader),'tamano_checkpoint_MB':size_mb(half_model),
         **profile(half_wrapper,test_loader,DEVICE)}
    print(f16)
    df=pd.concat([df,pd.DataFrame([f16])],ignore_index=True)
    df.to_csv(ROOT/'resultados_e2.csv',index=False)
    del half_wrapper,half_model
    torch.cuda.empty_cache()
else:
    print('No hay GPU; no se ejecuta el experimento FP16 real.')


## 5. Reproducibilidad y exportación
Guarda tablas, gráficos, configuración y checkpoint entrenado localmente. En Colab descarga todo al finalizar o monta Google Drive (no se suben pesos grandes al GitHub). Completar el log con resultados efectivamente observados, hipótesis aceptada/rechazada y limitaciones.

In [ ]:
config={'seed':SEED,'pytorch':torch.__version__,'torchvision':torchvision.__version__,'hardware':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else platform.processor(),
        'device':str(DEVICE),'weights':'ViT_B_16_Weights.IMAGENET1K_V1','input_shape':[3,224,224],
        'normalization':'ImageNet mean/std','N_TRAIN':N_TRAIN,'VAL':N_VAL_EVAL,'TEST':N_TEST_EVAL,'validation_split_total':len(val_ids),'test_split_total':len(raw_test),'run_type':'exploratorio_no_final','BATCH':BATCH,'EPOCHS':EPOCHS,
        'train_last_blocks':2,'quant_method':'per-tensor symmetric weight fake-quant dequant, 8-bit (no integer execution)','commit':'registrar hash del repo después de subir'}
(ROOT/'run_config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False))
import zipfile
with zipfile.ZipFile('grupo11_e2_resultados.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in ROOT.glob('*'):
        if f.suffix!='.pt':z.write(f,arcname=f.name)
print('Archivos en',ROOT.resolve());print('ZIP: grupo11_e2_resultados.zip')
